# GSE221156 human pancreatic islets (stage 2): donors, chemistry, clustering and cell-type annotation

Reads the quality-controlled object from stage 1 and does, in the standard Seurat workflow:

1. **Samples and donors:** one table of samples and donors (chemistry, disease group, BMI, age), counts per chemistry and group, and an **intersection plot** of disease group, chemistry and lean status.
2. **Quality control summary:** cells before and after filtering for every sample, and the differences between V2 and V3.
3. **Clustering:** normalise, variable genes, PCA, **elbow plot**, integrate across donors, cluster, UMAP **before annotation** (by cluster, sample, chemistry and disease group).
4. **Annotation** with markers supported by the Human Protein Atlas, UMAP **after annotation**, marker dot plot and feature plots, and cell counts and proportions overall and by donor, chemistry and group.

**How chemistry is handled.** Each donor was run on one kit (V2 or V3), so a donor sits wholly inside one chemistry. Integrating across **donors** therefore also removes the V2/V3 kit difference.
The notebook shows UMAPs before and after integration coloured by chemistry, and checks whether any cluster is made of one chemistry only.
Disease groups are never regressed out. Chemistry is a covariate in the stage 3 statistics, which are also run within V2 and within V3.

**Group design (donors with cells in the object).** Disease and lean status overlap very little: all lean donors are V3, none are PD, and only Islet84 (pool-only) is lean among T2D.
So the "intersection" of lean, pre-diabetic and diabetic is mostly empty by design; the plot and table in section 1 show exactly which combinations exist.
I tested the code on small made-up data only. Read each table before trusting it.

## 1. Setup

**Step:** Load the packages, set the paths and options, and read the object saved by stage 1.

In [ ]:
library(Seurat)
library(Matrix)
library(ggplot2)

out_dir <- "GSE221156_results"
qc_file <- file.path(out_dir, "GSE221156_seurat_QC.rds")

analysis_assay     <- "decontX"   # counts used for clustering and annotation ("RNA" = the original counts)
n_variable_genes   <- 2000
max_pcs            <- 30          # at most this many PCs are used
cluster_resolution <- 0.3
min_cells_batch    <- 100         # donors with fewer cells are grouped together for integration
hpa_markers_csv    <- ""          # optional: your own Human Protein Atlas table (columns cell_type, gene)

stage_dir <- file.path(out_dir, "stage2")
dir.create(stage_dir, showWarnings = FALSE, recursive = TRUE)
set.seed(1)

obj <- readRDS(qc_file)
if (!analysis_assay %in% Assays(obj)) analysis_assay <- "RNA"
DefaultAssay(obj) <- analysis_assay
cat("assay used:", analysis_assay, "\n")
obj

## 2. Samples and donors

**Step:** One table per donor and one per sample (library). A pooled library has one row per donor assigned to it.

In [ ]:
meta <- obj@meta.data

donor_table <- unique(meta[, c("donor", "group", "chemistry", "sex", "age", "bmi", "bmi_class", "lean_status", "hba1c")])
stopifnot(!anyDuplicated(donor_table$donor))               # each donor sits in one chemistry
donor_table$cells <- as.vector(table(meta$donor)[donor_table$donor])
donor_table <- donor_table[order(donor_table$chemistry, donor_table$group, donor_table$donor), ]

sample_table <- unique(meta[, c("library_id", "donor", "chemistry", "group", "donor_assignment")])
sample_table$cells <- as.vector(table(paste(meta$library_id, meta$donor))[paste(sample_table$library_id, sample_table$donor)])
sample_table <- merge(sample_table, donor_table[, c("donor", "age", "bmi", "sex", "lean_status")], by = "donor")

write.csv(donor_table,  file.path(stage_dir, "donor_table.csv"),  row.names = FALSE)
write.csv(sample_table, file.path(stage_dir, "sample_table.csv"), row.names = FALSE)
head(donor_table, 8)

**Step:** Count the samples, donors and cells in each chemistry and disease group.

In [ ]:
donor_table$donors <- 1
donor_table$lean   <- donor_table$lean_status == "Lean"
count_table <- aggregate(cbind(donors, lean, cells) ~ chemistry + group, data = donor_table, FUN = sum)

library_counts <- aggregate(library_id ~ chemistry + group, data = unique(sample_table[, c("library_id", "chemistry", "group")]), FUN = length)
names(library_counts)[3] <- "samples"
count_table <- merge(count_table, library_counts, by = c("chemistry", "group"))
write.csv(count_table, file.path(stage_dir, "counts_chemistry_group.csv"), row.names = FALSE)

count_table[, c("chemistry", "group", "donors", "samples", "cells", "lean")]
xtabs(donors ~ group + chemistry, data = count_table)      # donors only

*What you should see:* Donors per cell of the design. Expect V2 to hold no lean donors, and PD to have a single donor in V2 (Islet40). The totals depend on which pooled donors could be assigned in stage 1.

**Step:** The intersection plot. Each donor belongs to exactly one disease group and one chemistry, and is lean or not. The bars count the donors in every combination that exists.

In [ ]:
sets <- data.frame(
  ND   = donor_table$group == "ND",  PD = donor_table$group == "PD", T2D = donor_table$group == "T2D",
  V2   = donor_table$chemistry == "V2", V3 = donor_table$chemistry == "V3",
  Lean = donor_table$lean_status == "Lean"
)
combination <- apply(sets, 1, function(x) paste(names(sets)[x], collapse = " + "))
intersections <- as.data.frame(table(combination), stringsAsFactors = FALSE)
intersections <- intersections[order(intersections$Freq), ]
intersections$combination <- factor(intersections$combination, levels = intersections$combination)
write.csv(intersections, file.path(stage_dir, "intersections.csv"), row.names = FALSE)

p_bars <- ggplot(intersections, aes(combination, Freq)) +
  geom_col(fill = "steelblue") + geom_text(aes(label = Freq), hjust = -0.3) + coord_flip() +
  labs(title = "Donors in each combination of disease group, chemistry and lean status", x = NULL, y = "donors") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) + theme_bw()
p_bars
ggsave(file.path(stage_dir, "intersection_bars.png"), p_bars, width = 8, height = 5, dpi = 150)

*What you should see:* Bars such as ND + V3 + Lean and T2D + V2. Combinations with no donors do not appear: PD + Lean and V2 + Lean are absent.

**Step:** The same counts as a grid of disease group by chemistry (donors in each cell), the simplest picture of the design.

In [ ]:
grid <- aggregate(donors ~ group + chemistry, data = donor_table, FUN = sum)
p_grid <- ggplot(grid, aes(chemistry, group, fill = donors)) + geom_tile(colour = "white") +
  geom_text(aes(label = donors), size = 6) + scale_fill_gradient(low = "grey95", high = "steelblue") +
  labs(title = "Donors by disease group and chemistry", x = NULL, y = NULL) + theme_minimal()
p_grid
ggsave(file.path(stage_dir, "design_grid.png"), p_grid, width = 5, height = 4, dpi = 150)

## 3. Quality control: cells before and after filtering

**Step:** Read the exclusion log from stage 1: cells before and after filtering for every sample, and why cells were removed.

In [ ]:
log_file <- file.path(out_dir, "qc", "exclusion_log.csv")
exclusion_log <- read.csv(log_file)
names(exclusion_log)[names(exclusion_log) == "cells"] <- "before"
exclusion_log$sample <- paste0(exclusion_log$library_id, " (", exclusion_log$donor, ")")
exclusion_log <- exclusion_log[order(exclusion_log$chemistry, exclusion_log$group, exclusion_log$donor), ]
write.csv(exclusion_log, file.path(stage_dir, "cells_before_after_per_sample.csv"), row.names = FALSE)
exclusion_log[, c("sample", "chemistry", "group", "before", "low_genes", "low_umis", "high_mito", "doublet", "kept", "percent_kept")]

*What you should see:* One row per sample with cells before filtering, cells removed for each reason, cells kept and the percentage kept.

**Step:** Summarise by chemistry and group, and draw cells before and after for every sample.

In [ ]:
qc_summary <- aggregate(cbind(before, kept) ~ chemistry + group, data = exclusion_log, FUN = sum)
qc_summary$percent_kept <- round(100 * qc_summary$kept / qc_summary$before, 1)
qc_summary

long <- rbind(
  data.frame(sample = exclusion_log$sample, chemistry = exclusion_log$chemistry, stage = "before", cells = exclusion_log$before),
  data.frame(sample = exclusion_log$sample, chemistry = exclusion_log$chemistry, stage = "after",  cells = exclusion_log$kept)
)
long$stage <- factor(long$stage, levels = c("before", "after"))
p_qc <- ggplot(long, aes(sample, cells, fill = stage)) + geom_col(position = "dodge") +
  facet_grid(. ~ chemistry, scales = "free_x", space = "free_x") +
  labs(title = "Cells per sample before and after QC", x = NULL) + theme_bw() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
p_qc
ggsave(file.path(stage_dir, "cells_before_after.png"), p_qc, width = 14, height = 5, dpi = 130)

**Step:** Are V2 and V3 different? Compare the QC metrics between chemistries using one value per donor (the median), so each donor counts once.

In [ ]:
donor_qc <- aggregate(cbind(nCount_RNA, nFeature_RNA, percent.mt) ~ donor + chemistry, data = obj@meta.data, FUN = median)
aggregate(cbind(nCount_RNA, nFeature_RNA, percent.mt) ~ chemistry, data = donor_qc, FUN = median)

for (metric in c("nCount_RNA", "nFeature_RNA", "percent.mt")) {
  p <- wilcox.test(donor_qc[[metric]] ~ donor_qc$chemistry, exact = FALSE)$p.value
  cat(metric, ": V2 vs V3 p =", signif(p, 3), "\n")
}
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), group.by = "chemistry", pt.size = 0, ncol = 3)

*What you should see:* Median depth, genes and mitochondrial percentage per chemistry, and a donor-level p-value for each. A small p-value says the chemistries differ, which is why integration and the covariate matter.

## 4. Normalise, elbow plot, integrate, cluster
Standard Seurat v5 workflow. The data are split into one layer per donor, normalised, reduced by PCA, and the donors are **integrated** (RPCA). Because every donor belongs to one chemistry, this aligns V2 and V3 as well.
The disease group is not used anywhere in this step.

**Step:** Helper functions: choose the number of PCs from the elbow, and group very small donors together so integration can run.

In [ ]:
number_of_pcs <- function(stdev) {          # the published atlas's rule for significant PCs
  percent <- stdev / sum(stdev) * 100
  cumulative <- cumsum(percent)
  first  <- which(cumulative > 90 & percent < 5)[1]
  second <- sort(which((percent[-length(percent)] - percent[-1]) > 0.05), decreasing = TRUE)[1] + 1
  min(first, second, na.rm = TRUE)
}

**Step:** Split by donor, normalise, find variable genes, scale and run PCA. Draw the **elbow plot**.

In [ ]:
obj$batch <- obj$donor
small_donors <- names(which(table(obj$batch) < min_cells_batch))
obj$batch[obj$batch %in% small_donors] <- "small_donors"
obj[[analysis_assay]] <- split(obj[[analysis_assay]], f = obj$batch)

obj <- NormalizeData(obj)
obj <- FindVariableFeatures(obj, nfeatures = n_variable_genes)
obj <- ScaleData(obj)
obj <- RunPCA(obj, npcs = 50)

n_pcs <- min(number_of_pcs(Stdev(obj, reduction = "pca")), max_pcs)
cat("PCs used:", n_pcs, "\n")
elbow <- ElbowPlot(obj, ndims = 50) + geom_vline(xintercept = n_pcs, colour = "red", linetype = 2) +
  ggtitle(paste("Elbow plot:", n_pcs, "PCs used"))
elbow
ggsave(file.path(stage_dir, "elbow_plot.png"), elbow, width = 6, height = 4, dpi = 150)

*What you should see:* A curve that bends and flattens, with a red line at the chosen number of PCs (at most 30).

**Step:** UMAP **before integration**, to see whether V2 and V3 separate. Then integrate across donors.

In [ ]:
obj <- RunUMAP(obj, reduction = "pca", dims = 1:n_pcs, reduction.name = "umap.unintegrated")

integrated <- NULL
k_weight <- 100
while (is.null(integrated) && k_weight >= 5) {
  integrated <- tryCatch(
    IntegrateLayers(obj, method = RPCAIntegration, orig.reduction = "pca", new.reduction = "integrated.rpca", k.weight = k_weight, verbose = FALSE),
    error = function(e) NULL)
  if (is.null(integrated)) k_weight <- floor(k_weight / 2)      # small donors need a smaller k.weight
}
stopifnot(!is.null(integrated))
obj <- integrated
rm(integrated)
obj[[analysis_assay]] <- JoinLayers(obj[[analysis_assay]])

**Step:** Find neighbours, cluster, and make the UMAP of the integrated data.

In [ ]:
obj <- FindNeighbors(obj, reduction = "integrated.rpca", dims = 1:n_pcs)
obj <- FindClusters(obj, resolution = cluster_resolution)
obj <- RunUMAP(obj, reduction = "integrated.rpca", dims = 1:n_pcs, reduction.name = "umap")
table(obj$seurat_clusters)

### UMAPs before annotation

**Step:** Did integration remove the chemistry difference? The same UMAP coloured by chemistry, before and after integration.

In [ ]:
p_before <- DimPlot(obj, reduction = "umap.unintegrated", group.by = "chemistry") + ggtitle("Before integration")
p_after  <- DimPlot(obj, reduction = "umap", group.by = "chemistry") + ggtitle("After integration")
p_chem <- p_before + p_after
p_chem
ggsave(file.path(stage_dir, "umap_chemistry_before_after_integration.png"), p_chem, width = 13, height = 5.5, dpi = 130)

*What you should see:* Before: V2 and V3 partly separate. After: they overlap within each cell population.

**Step:** UMAPs of the integrated data coloured by cluster, sample (donor), chemistry and disease group.

In [ ]:
p1 <- DimPlot(obj, group.by = "seurat_clusters", label = TRUE) + ggtitle("Clusters")
p2 <- DimPlot(obj, group.by = "donor") + NoLegend() + ggtitle("Sample (donor)")
p3 <- DimPlot(obj, group.by = "chemistry") + ggtitle("Chemistry")
p4 <- DimPlot(obj, group.by = "group") + ggtitle("Disease group")
umaps_before <- (p1 + p2) / (p3 + p4)
umaps_before
ggsave(file.path(stage_dir, "umap_before_annotation.png"), umaps_before, width = 14, height = 11, dpi = 120)

*What you should see:* Four panels. The donor, chemistry and group colours should be mixed within clusters, not one colour per cluster.

**Step:** Cells by cluster, sample, chemistry and disease group. A cluster made of one chemistry or of one donor deserves a look.

In [ ]:
by_chemistry <- table(obj$seurat_clusters, obj$chemistry)
by_group     <- table(obj$seurat_clusters, obj$group)
cluster_summary <- data.frame(
  cluster = rownames(by_chemistry), cells = rowSums(by_chemistry),
  percent_V2 = round(100 * by_chemistry[, "V2"] / rowSums(by_chemistry), 1),
  donors = as.vector(tapply(obj$donor, obj$seurat_clusters, function(x) length(unique(x))))
)
rownames(cluster_summary) <- NULL
cluster_summary <- cbind(cluster_summary, as.data.frame.matrix(round(100 * prop.table(by_group, 1), 1)))   # % of the cluster in each disease group
write.csv(cluster_summary, file.path(stage_dir, "cluster_composition.csv"), row.names = FALSE)
cluster_summary

overall_v2 <- round(100 * mean(obj$chemistry == "V2"), 1)
cat("\nV2 is", overall_v2, "% of all cells. Clusters above 90% or below 10% of V2 (and over 200 cells):\n")
print(cluster_summary[cluster_summary$cells > 200 & (cluster_summary$percent_V2 > 90 | cluster_summary$percent_V2 < 10), ])

*What you should see:* Percent V2 per cluster should stay near the overall value. Extreme clusters can be real cell types that one chemistry captured better, or leftover batch effect.

## 5. Cell-type annotation with Human Protein Atlas markers
For each cluster the notebook finds its **differentially expressed genes**, scores it against marker panels for the islet cell types, and gives it a name only if the best score is clear and ahead of the runner-up by a margin; otherwise it stays `Unassigned`.

*Be careful.* **The marker lists below were written from my memory of the Human Protein Atlas pancreas data and the paper's own markers, not downloaded.** Check them against proteinatlas.org, or replace them with your own table through `hpa_markers_csv`.
**PECAM1, CDH5, ERG and the other genes you test are never used to name clusters**, so they cannot bias their own result; they are only drawn on the plots.

**Step:** The marker panels (islet cell types, from the Human Protein Atlas and the paper) and the genes excluded from naming.

In [ ]:
hpa_panels <- list(
  Beta           = c("INS", "IAPP", "MAFA", "NKX6-1", "PDX1", "G6PC2", "ADCYAP1"),
  Alpha          = c("GCG", "ARX", "IRX2", "TTR", "LOXL4", "GC"),
  Delta          = c("SST", "HHEX", "RBP4", "LEPR", "BCHE"),
  PP             = c("PPY", "SERTM1", "ETV1", "CARTPT"),
  Acinar         = c("PRSS1", "CPA1", "CELA3A", "CTRB1", "REG1A", "PNLIP", "CPB1"),
  Ductal         = c("KRT19", "SOX9", "CFTR", "MMP7", "SPP1", "ANXA4", "KRT7"),
  Stellate       = c("COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRB", "RGS5", "FABP4"),
  Endothelial    = c("PLVAP", "VWF", "KDR", "EMCN", "ESAM", "FLT1", "RAMP2", "CLDN5", "ECSCR"),
  Immune         = c("PTPRC", "CD68", "C1QC", "C1QA", "CD74", "HLA-DRA", "MS4A7", "CD3E"),
  Mast           = c("TPSAB1", "TPSB2", "KIT", "CPA3"),
  Schwann        = c("NGFR", "S100B", "PLP1", "SOX10", "MPZ"),
  Proliferating  = c("TOP2A", "MKI67", "CENPF", "STMN1")
)
if (hpa_markers_csv != "" && file.exists(hpa_markers_csv)) {
  table_in <- read.csv(hpa_markers_csv, stringsAsFactors = FALSE)
  hpa_panels <- split(table_in$gene, table_in$cell_type)
}

genes_of_interest <- c("ERG", "PECAM1", "CDH5", "NOS3", "VCAM1", "ICAM1", "HIF1A", "ANGPT2")
hpa_panels <- lapply(hpa_panels, function(g) intersect(setdiff(g, genes_of_interest), rownames(obj)))
hpa_panels <- hpa_panels[lengths(hpa_panels) >= 2]     # a panel needs at least 2 genes: one gene alone gives unreliable scores
sapply(hpa_panels, length)             # panel genes found in the data

*What you should see:* The number of panel genes present for each cell type. Epsilon cells (GHRL) are rare and have too few markers to name a cluster; GHRL is shown on the feature plots only.

**Step:** Differentially expressed genes of every cluster (one cluster against all others). Slow on large data; install `presto` to speed it up. At most 500 cells per cluster are used here, **only to find markers**.

In [ ]:
markers <- FindAllMarkers(obj, assay = analysis_assay, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.5, max.cells.per.ident = 500)
write.csv(markers, file.path(stage_dir, "cluster_markers_all.csv"), row.names = FALSE)

top_markers <- do.call(rbind, lapply(split(markers, markers$cluster), function(d) head(d[order(-d$avg_log2FC), ], 10)))
write.csv(top_markers, file.path(stage_dir, "cluster_markers_top10.csv"), row.names = FALSE)
head(top_markers[, c("cluster", "gene", "avg_log2FC", "pct.1", "pct.2")], 12)

**Step:** Score every cluster against every panel (mean z-score of the panel genes across clusters), then name each cluster.

In [ ]:
avg <- AverageExpression(obj, assays = analysis_assay, features = unique(unlist(hpa_panels)), group.by = "seurat_clusters", layer = "data")[[analysis_assay]]
avg <- log1p(as.matrix(avg))
colnames(avg) <- sub("^g", "", colnames(avg))
z <- t(scale(t(avg)))
z[is.na(z)] <- 0

scores <- sapply(hpa_panels, function(genes) colMeans(z[genes, , drop = FALSE]))    # clusters x cell types
min_score  <- 0.25
min_margin <- 0.25

annotation <- do.call(rbind, lapply(rownames(scores), function(cl) {
  ranked <- sort(scores[cl, ], decreasing = TRUE)
  data.frame(cluster = cl, best = names(ranked)[1], score = round(ranked[1], 2), runner_up = names(ranked)[2],
             margin = round(ranked[1] - ranked[2], 2),
             label = ifelse(ranked[1] >= min_score & ranked[1] - ranked[2] >= min_margin, names(ranked)[1], "Unassigned"))
}))
rownames(annotation) <- NULL

# the cluster's own top markers that belong to the winning panel
annotation$markers_in_panel <- sapply(annotation$cluster, function(cl) {
  top <- top_markers$gene[top_markers$cluster == cl]
  paste(intersect(top, hpa_panels[[annotation$best[annotation$cluster == cl]]]), collapse = ",")
})
write.csv(annotation, file.path(stage_dir, "cluster_annotation.csv"), row.names = FALSE)
annotation

*What you should see:* One row per cluster: best panel, score, runner-up, margin, final label, and which of the cluster's top markers belong to that panel. Check the rows labelled Unassigned or with a small margin.

**Step:** Write the names into the object.

In [ ]:
obj$cell_type <- annotation$label[match(as.character(obj$seurat_clusters), annotation$cluster)]
sort(table(obj$cell_type), decreasing = TRUE)

### UMAP and markers after annotation

**Step:** UMAP **after annotation**, also split by chemistry and by disease group.

In [ ]:
p_annot <- DimPlot(obj, group.by = "cell_type", label = TRUE, repel = TRUE) + ggtitle("Annotated cell types")
p_split_chem  <- DimPlot(obj, group.by = "cell_type", split.by = "chemistry") + ggtitle("By chemistry")
p_split_group <- DimPlot(obj, group.by = "cell_type", split.by = "group") + ggtitle("By disease group")
p_annot
p_split_chem
p_split_group
ggsave(file.path(stage_dir, "umap_annotated.png"), p_annot, width = 9, height = 7, dpi = 150)
ggsave(file.path(stage_dir, "umap_annotated_by_chemistry.png"), p_split_chem, width = 13, height = 6, dpi = 130)
ggsave(file.path(stage_dir, "umap_annotated_by_group.png"), p_split_group, width = 16, height = 6, dpi = 130)

**Step:** Marker dot plot (the top markers of each panel plus PECAM1, CDH5 and ERG, which are shown but not used for naming) and feature plots of key markers.

In [ ]:
dot_genes <- unique(c(unlist(lapply(hpa_panels, head, 2)), "PECAM1", "CDH5", "ERG"))
dot_genes <- intersect(dot_genes, rownames(obj))
p_dot <- DotPlot(obj, features = dot_genes, group.by = "cell_type", assay = analysis_assay) + RotatedAxis() + ggtitle("Marker genes by cell type")
p_dot
ggsave(file.path(stage_dir, "dotplot_markers.png"), p_dot, width = 15, height = 6, dpi = 130)

key_genes <- intersect(c("INS", "GCG", "SST", "PPY", "GHRL", "PRSS1", "KRT19", "COL1A1", "PLVAP", "PTPRC", "ERG"), rownames(obj))
p_feat <- FeaturePlot(obj, features = key_genes, ncol = 4, order = TRUE)
p_feat
ggsave(file.path(stage_dir, "featureplot_key_markers.png"), p_feat, width = 16, height = 11, dpi = 100)

### Cell counts and proportions

**Step:** Number and proportion of cells of each type: overall, then by chemistry and by disease group.

In [ ]:
overall <- as.data.frame(table(cell_type = obj$cell_type))
overall$percent <- round(100 * overall$Freq / sum(overall$Freq), 2)
overall <- overall[order(-overall$Freq), ]
overall

by_chem  <- table(obj$cell_type, obj$chemistry)
by_group <- table(obj$cell_type, obj$group)
by_both  <- table(obj$cell_type, paste(obj$group, obj$chemistry, sep = "_"))
counts_out <- list(overall = overall, chemistry = by_chem, group = by_group, group_chemistry = by_both)
cat("\nPercent of each chemistry's cells:\n");  round(100 * prop.table(by_chem, 2), 1)
cat("\nPercent of each disease group's cells:\n");  round(100 * prop.table(by_group, 2), 1)
cat("\nPercent of each group x chemistry:\n");  round(100 * prop.table(by_both, 2), 1)
write.csv(as.data.frame.matrix(by_both), file.path(stage_dir, "cell_counts_type_by_group_chemistry.csv"))

**Step:** By donor: cells of each type for every donor, as counts and as percent of the donor's cells.

In [ ]:
by_donor <- table(obj$donor, obj$cell_type)
percent_donor <- round(100 * prop.table(by_donor, 1), 1)
write.csv(as.data.frame.matrix(by_donor),      file.path(stage_dir, "cell_counts_by_donor.csv"))
write.csv(as.data.frame.matrix(percent_donor), file.path(stage_dir, "cell_percent_by_donor.csv"))

plot_data <- as.data.frame(percent_donor)
names(plot_data) <- c("donor", "cell_type", "percent")
plot_data <- merge(plot_data, donor_table[, c("donor", "group", "chemistry")], by = "donor")
p_comp <- ggplot(plot_data, aes(donor, percent, fill = cell_type)) + geom_col() +
  facet_grid(. ~ chemistry + group, scales = "free_x", space = "free_x") +
  labs(title = "Cell-type composition per donor", x = NULL, y = "% of cells") + theme_bw() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
p_comp
ggsave(file.path(stage_dir, "composition_per_donor.png"), p_comp, width = 16, height = 6, dpi = 120)

head(by_donor, 5)

**Step:** Bar graph of the number of cells per annotated type, and save the annotated object for stage 3.

In [ ]:
p_counts <- ggplot(overall, aes(reorder(cell_type, Freq), Freq)) + geom_col(fill = "steelblue") + coord_flip() +
  geom_text(aes(label = Freq), hjust = -0.1) + labs(title = "Cells per annotated type", x = NULL, y = "cells") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.12))) + theme_bw()
p_counts
ggsave(file.path(stage_dir, "cell_counts_per_type.png"), p_counts, width = 7, height = 5, dpi = 150)

saveRDS(obj, file.path(out_dir, "GSE221156_annotated.rds"))

## Next
Stage 3 (`GSE221156_stage3_endothelial_ERG_pseudobulk.ipynb`) takes the annotated object: the endothelial cluster and its markers, the eight genes of interest, ERG in every cell type, and the donor-level pseudobulk tests within V2, within V3 and combined.

**Limits.** The marker lists are from memory and need checking against the Human Protein Atlas. A cluster labelled `Unassigned` is a prompt to look at its markers, not a finding. This notebook describes composition; it does not test differences in cell-type proportions.